# Exploratory Data Analysis: Crime Hotspot Prediction

This notebook is the single source reference for every figure in the report's exploratory analysis.
Each section names the report Figure it reproduces, reads the corrected pipeline outputs from the Kedro
data layers, and saves the figure into `figures/` under the same filename used in the report.

Run the pipeline first (`kedro run --pipeline training_from_raw`) so the data layers hold the corrected
data, then run this notebook top to bottom.


## 1. Setup and data loading


In [ ]:
# Render every inline figure with a tight bounding box so long axis labels and
# titles are never clipped in the notebook display.
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight'}

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch

OKABE = ["#0072B2","#E69F00","#009E73","#D55E00","#CC79A7","#56B4E9","#F0E442","#000000","#999999"]
PRIMARY, SECONDARY, GOOD, INK, MUTE = "#1F4E5F", "#C44536", "#2A9D8F", "#1a1a1a", "#5a5a5a"
plt.rcParams.update({
    "figure.dpi":110, "savefig.dpi":150, "font.family":"DejaVu Sans", "font.size":11,
    "axes.titlesize":13, "axes.titleweight":"bold", "axes.grid":True, "axes.axisbelow":True,
    "grid.color":"#dcdcdc", "grid.linewidth":0.7, "axes.spines.top":False, "axes.spines.right":False,
    "figure.facecolor":"white", "axes.facecolor":"white", "legend.frameon":False})
os.makedirs("figures", exist_ok=True)
def save(fig, name):
    try: fig.tight_layout()
    except Exception: pass
    fig.savefig(f"figures/{name}", dpi=150, bbox_inches="tight", facecolor="white")
tk = FuncFormatter(lambda x,_: f"{x:,.0f}")

DATA = ".."  # notebook lives in <project>/notebooks
crime  = pd.read_excel(f"{DATA}/data/02_intermediate/crime_processed.xlsx")
master = pd.read_excel(f"{DATA}/data/03_primary/master_dataset.xlsx")
crime["date"]  = pd.to_datetime(crime["date"]);  crime["year"]  = crime["date"].dt.year
master["date"] = pd.to_datetime(master["date"]); master["year"] = master["date"].dt.year
SOCIO = ["population_density","poor_households","population_unemployment","population_education"]
SL = {"population_density":"Pop. density","poor_households":"Poor households",
      "population_unemployment":"Unemployment","population_education":"Education"}
order = crime.pivot_table(index="year", columns="Cluster", values="Crime Count", aggfunc="sum").mean().sort_values(ascending=False).index
print("crime panel:", crime.shape, "| master panel:", master.shape)
print("clusters (crime):", crime['Cluster'].nunique(), "| crime types:", crime['Type of Crime'].nunique(),
      "| years:", int(crime.year.min()), "to", int(crime.year.max()))
crime.head()


## 2. Data-quality checks and the Polokwane deduplication (report Figure 6)

The two police clusters Mankweng and Seshego both map to the Polokwane municipality, which creates two
rows per (date, crime type) for Polokwane. The pipeline aggregates on the full key
`(date, Cluster, Type of Crime)`, never on cluster alone, and **sums** the matching rows, so no count is
lost. The cell below reconstructs the before state from the raw files (via the pipeline's own
transformers) and confirms the total is preserved. This is the source for report Figure 6.


In [ ]:
import importlib.util, yaml
spec = importlib.util.spec_from_file_location(
    "tr", f"{DATA}/src/crime_hotspot_prediction_project/pipelines/data_preprocessing/transformers.py")
tr = importlib.util.module_from_spec(spec); spec.loader.exec_module(tr)
excl = yaml.safe_load(open(f"{DATA}/conf/base/parameters.yml"))["excluded_crime_types"]

raw = [pd.read_excel(f"{DATA}/data/01_raw/crime/crime_df{i}.xlsx") for i in (1,2,3)]
d = tr.preprocess().fit_transform(raw); d = tr.table_structure().fit_transform(d); d = tr.data_split().fit_transform(d)
d = tr.data_cleaning().fit_transform(d); d = tr.feature_engineering_crime().fit_transform(d); d = tr.crime_data_cleaning().fit_transform(d)
before = tr.exclude_crime_categories(excl).fit_transform(d)   # cluster mapped, aggregates removed, NOT yet deduped
after  = tr.aggregate_crime_duplicates().fit_transform(before) # groups on (date,Cluster,Type) and SUMS

print("aggregation key:", "['date','Cluster','Type of Crime']  (never cluster alone)")
print("total Crime Count before dedup:", int(before['Crime Count'].sum()))
print("total Crime Count after  dedup:", int(after['Crime Count'].sum()), " -> identical, summing preserves every incident")
print("duplicate (date,Cluster,Type) rows in the corrected panel:", int(after.duplicated(['date','Cluster','Type of Crime']).sum()))
print("clusters that had duplicates:", sorted(before[before.duplicated(['date','Cluster','Type of Crime'],keep=False)].Cluster.unique()))

# raw (uncorrected) crime, for the before/after counts in Figure 6
draw = tr.preprocess().fit_transform(raw); draw = tr.table_structure().fit_transform(draw); draw = tr.data_split().fit_transform(draw)
draw = tr.data_cleaning().fit_transform(draw); draw = tr.feature_engineering_crime().fit_transform(draw)
draw = tr.crime_data_cleaning().fit_transform(draw); draw = tr.date_filter(start_year=2008).fit_transform(draw)
panels = [("Duplicate\n(cluster,type,year) rows", int(draw.duplicated(['date','Cluster','Type of Crime']).sum()), 0),
          ("Distinct crime types", int(draw['Type of Crime'].nunique()), int(crime['Type of Crime'].nunique())),
          ("Crime rows", len(draw), len(crime))]
fig, axs = plt.subplots(1, 3, figsize=(11,3.4))
for ax,(t,b,a) in zip(axs, panels):
    ax.bar(["Before","After"], [b,a], color=[SECONDARY,GOOD], width=0.6, edgecolor="white")
    for i,v in enumerate([b,a]): ax.text(i,v,f"{v:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(t, fontsize=10.5); ax.set_ylim(0, max(b,a)*1.18)
fig.suptitle("Data-quality corrections, before vs after", fontweight="bold", y=1.03)
save(fig,"e13_dataquality.png"); plt.show()


## 3. Univariate structure of the target (report Figures 12, 13, 14)


In [ ]:
# Figure 12: distribution of Crime Count (linear and log)
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].hist(crime["Crime Count"], bins=60, color=PRIMARY, edgecolor="white", linewidth=0.3)
ax[0].set_title("Distribution of Crime Count (linear)"); ax[0].set_xlabel("Crime Count (per cluster, crime type, year)"); ax[0].set_ylabel("Number of records"); ax[0].yaxis.set_major_formatter(tk)
ax[1].hist(np.log1p(crime["Crime Count"].clip(lower=0)), bins=60, color=GOOD, edgecolor="white", linewidth=0.3)
ax[1].set_title("Distribution of log(1 + Crime Count)"); ax[1].set_xlabel("log(1 + Crime Count)"); ax[1].set_ylabel("Number of records")
ax[0].annotate(f"Right-skewed\nskewness = {crime['Crime Count'].skew():.2f}\nmedian = {crime['Crime Count'].median():.0f}\nmean = {crime['Crime Count'].mean():.0f}",
               xy=(0.40,0.60), xycoords="axes fraction", fontsize=10, color=INK, ha="left", va="center",
               bbox=dict(boxstyle="round,pad=0.4", fc="#f4f4f4", ec="#cccccc"))
fig.suptitle("Crime Count is strongly right-skewed", fontweight="bold", y=1.02)
save(fig,"e01_crimecount_dist.png"); plt.show()


In [ ]:
# Figure 13: Crime Count by crime type (top 15 by median)
top = crime.groupby("Type of Crime")["Crime Count"].median().sort_values(ascending=False).head(15).index[::-1]
fig, ax = plt.subplots(figsize=(9,6.5))
bp = ax.boxplot([crime[crime["Type of Crime"]==t]["Crime Count"] for t in top], vert=False, patch_artist=True, widths=0.6,
                flierprops=dict(marker="o", markersize=2, markerfacecolor=MUTE, markeredgecolor="none", alpha=0.4))
for b in bp["boxes"]: b.set(facecolor=PRIMARY, alpha=0.75)
for m in bp["medians"]: m.set(color="white", linewidth=1.6)
ax.set_yticklabels(top, fontsize=9); ax.set_xlabel("Crime Count (per cluster-year)"); ax.set_ylabel("Crime type")
ax.set_title("Crime Count by crime type (top 15 by median)")
save(fig,"e02_box_by_type.png"); plt.show()


In [ ]:
# Figure 14: distributions of the four socioeconomic indicators
fig, axs = plt.subplots(2,2, figsize=(10,7))
for ax, s in zip(axs.ravel(), SOCIO):
    ax.hist(master[s], bins=40, color=PRIMARY, edgecolor="white", linewidth=0.3)
    ax.set_title(SL[s], fontsize=11); ax.set_xlabel(SL[s]); ax.set_ylabel("Records"); ax.xaxis.set_major_formatter(tk)
fig.suptitle("Distributions of the four socioeconomic indicators", fontweight="bold", y=1.01)
save(fig,"e16_socio_dist.png"); plt.show()


## 4. Temporal structure (report Figures 15, 16, 17, 18)


In [ ]:
# Figure 15: total annual trend
tot = crime.groupby("year")["Crime Count"].sum()
fig, ax = plt.subplots(figsize=(9,4.2))
ax.plot(tot.index, tot.values, marker="o", color=PRIMARY, lw=2.2); ax.fill_between(tot.index, tot.values, color=PRIMARY, alpha=0.08)
ax.set_title("Total recorded Crime Count over time"); ax.set_xlabel("Year"); ax.set_ylabel("Total Crime Count"); ax.yaxis.set_major_formatter(tk)
save(fig,"e03_total_trend.png"); plt.show()


In [ ]:
# Figure 16: annual Crime Count by cluster
piv = crime.pivot_table(index="year", columns="Cluster", values="Crime Count", aggfunc="sum")
fig, ax = plt.subplots(figsize=(9.5,5))
for i, cl in enumerate(order):
    ax.plot(piv.index, piv[cl], lw=2 if i<4 else 1, color=OKABE[i%9] if i<8 else "#c0c0c0", label=cl if i<8 else None, alpha=0.95 if i<8 else 0.6)
ax.set_title("Annual Crime Count by cluster"); ax.set_xlabel("Year"); ax.set_ylabel("Crime Count"); ax.yaxis.set_major_formatter(tk); ax.legend(ncol=2, fontsize=8)
save(fig,"e04_by_cluster.png"); plt.show()


In [ ]:
# Figure 17: crime type composition over time (top 8 types)
tt = crime.groupby("Type of Crime")["Crime Count"].sum().sort_values(ascending=False).head(8).index
comp = crime[crime["Type of Crime"].isin(tt)].pivot_table(index="year", columns="Type of Crime", values="Crime Count", aggfunc="sum").fillna(0)[tt]
fig, ax = plt.subplots(figsize=(9.5,5))
ax.stackplot(comp.index, [comp[c] for c in comp.columns], labels=list(comp.columns), colors=OKABE[:len(comp.columns)], alpha=0.9)
ax.set_title("Composition of Crime Count over time (top 8 types)"); ax.set_xlabel("Year"); ax.set_ylabel("Crime Count"); ax.yaxis.set_major_formatter(tk); ax.legend(ncol=2, fontsize=8)
save(fig,"e06_composition.png"); plt.show()


In [ ]:
# Figure 18: year-on-year percentage change by cluster
yoy = crime.groupby(["year","Cluster"])["Crime Count"].sum().groupby("Cluster").pct_change().mul(100).reset_index().dropna()
fig, ax = plt.subplots(figsize=(9.5,5))
for i, cl in enumerate(order[:8]):
    d = yoy[yoy.Cluster==cl]; ax.plot(d.year, d["Crime Count"], marker="o", ms=3, lw=1.4, color=OKABE[i%9], label=cl)
ax.axhline(0, color=MUTE, lw=0.9); ax.set_title("Year-on-year percentage change by cluster"); ax.set_xlabel("Year"); ax.set_ylabel("Change (%)"); ax.legend(ncol=2, fontsize=8)
save(fig,"e12_yoy.png"); plt.show()


## 5. Spatial and cluster structure (report Figures 19, 20, 24e)


In [ ]:
# Figure 19: mean annual Crime Count by cluster
mean_annual = crime.groupby(["Cluster","year"])["Crime Count"].sum().groupby("Cluster").mean().sort_values()
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(mean_annual.index, mean_annual.values, color=PRIMARY, edgecolor="white")
for i,(k,v) in enumerate(mean_annual.items()): ax.text(v,i,f" {v:,.0f}", va="center", fontsize=9)
ax.set_title("Mean annual Crime Count by cluster"); ax.set_xlabel("Mean annual Crime Count"); ax.xaxis.set_major_formatter(tk)
save(fig,"e05_top_clusters.png"); plt.show()


In [ ]:
# Figure 20: cluster x crime type heatmap
tt = crime.groupby("Type of Crime")["Crime Count"].sum().sort_values(ascending=False).head(15).index
hm = crime[crime["Type of Crime"].isin(tt)].pivot_table(index="Cluster", columns="Type of Crime", values="Crime Count", aggfunc="mean").fillna(0)[tt]
fig, ax = plt.subplots(figsize=(11,5.5)); im = ax.imshow(hm.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(hm.columns))); ax.set_xticklabels(hm.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(hm.index))); ax.set_yticklabels(hm.index, fontsize=9); ax.grid(False)
ax.set_title("Mean annual Crime Count by cluster and crime type"); fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
save(fig,"e07_heatmap_cluster_type.png"); plt.show()


In [ ]:
# Figure 24e: distribution of Crime Count by cluster
co = crime.groupby("Cluster")["Crime Count"].median().sort_values().index
fig, ax = plt.subplots(figsize=(9,5))
bp = ax.boxplot([crime[crime.Cluster==c]["Crime Count"] for c in co], vert=False, patch_artist=True, widths=0.6,
                flierprops=dict(marker="o", markersize=2, markerfacecolor=MUTE, markeredgecolor="none", alpha=0.4))
for b in bp["boxes"]: b.set(facecolor=PRIMARY, alpha=0.75)
for m in bp["medians"]: m.set(color="white", linewidth=1.6)
ax.set_yticklabels(co, fontsize=9); ax.set_xlabel("Crime Count (per crime type-year)"); ax.set_ylabel("Cluster")
ax.set_title("Distribution of Crime Count by cluster")
save(fig,"e21_crime_by_cluster.png"); plt.show()


## 6. Bivariate structure and multicollinearity (report Figures 21, 22, 23, 24, 24b, 24c)

This is where the model choice is justified by the data: the socioeconomic indicators are strongly
collinear with one another (high VIF) but only weakly and non-linearly correlated with Crime Count.


In [ ]:
# Figure 24b: correlation matrix, Crime Count and indicators
cols = ["Crime Count"] + SOCIO; cc = master[cols].corr(); labels = ["Crime Count"] + [SL[s] for s in SOCIO]
fig, ax = plt.subplots(figsize=(6.2,5.2)); im = ax.imshow(cc.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cc))); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(cc))); ax.set_yticklabels(labels, fontsize=9); ax.grid(False)
for i in range(len(cc)):
    for j in range(len(cc)):
        ax.text(j,i,f"{cc.values[i,j]:.2f}", ha="center", va="center", color="white" if abs(cc.values[i,j])>0.6 else INK, fontsize=9, fontweight="bold")
ax.set_title("Correlation: Crime Count and socioeconomic indicators"); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
save(fig,"e19_full_corr.png"); plt.show()


In [ ]:
# Figure 21: correlation among the four indicators
cc2 = master[SOCIO].rename(columns=SL).corr()
fig, ax = plt.subplots(figsize=(6,5)); im = ax.imshow(cc2.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cc2))); ax.set_xticklabels(cc2.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(cc2))); ax.set_yticklabels(cc2.index, fontsize=9); ax.grid(False)
for i in range(len(cc2)):
    for j in range(len(cc2)):
        ax.text(j,i,f"{cc2.values[i,j]:.2f}", ha="center", va="center", color="white" if abs(cc2.values[i,j])>0.6 else INK, fontsize=10, fontweight="bold")
ax.set_title("Correlation among socioeconomic indicators"); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
save(fig,"e08_socio_corr.png"); plt.show()


In [ ]:
# Figure 23: correlation of each indicator with Crime Count
cs = pd.Series({SL[s]: master[["Crime Count",s]].corr().iloc[0,1] for s in SOCIO}).sort_values()
fig, ax = plt.subplots(figsize=(8,3.6))
ax.barh(cs.index, cs.values, color=[SECONDARY if v<0 else PRIMARY for v in cs.values], edgecolor="white")
for i,(k,v) in enumerate(cs.items()): ax.text(v,i,f" {v:.3f}", va="center", fontsize=10)
ax.axvline(0, color=MUTE, lw=0.8); ax.set_xlim(-0.5,0.5)
ax.set_title("Pearson correlation of each indicator with Crime Count"); ax.set_xlabel("Pearson r"); ax.set_ylabel("Indicator")
save(fig,"e09_socio_vs_crime.png"); plt.show()


In [ ]:
# Figure 24: scatter of each indicator vs Crime Count with OLS fit
fig, axs = plt.subplots(2,2, figsize=(10,8))
for ax, s in zip(axs.ravel(), SOCIO):
    x, y = master[s], master["Crime Count"]
    ax.scatter(x, y, s=8, alpha=0.35, color=PRIMARY, edgecolors="none")
    b, a = np.polyfit(x, y, 1); xs = np.linspace(x.min(), x.max(), 50); ax.plot(xs, a+b*xs, color=SECONDARY, lw=2)
    ax.set_title(f"{SL[s]}  (r = {np.corrcoef(x,y)[0,1]:.3f})"); ax.set_xlabel(SL[s]); ax.set_ylabel("Crime Count")
    ax.xaxis.set_major_formatter(tk); ax.yaxis.set_major_formatter(tk)
fig.suptitle("Socioeconomic indicators vs Crime Count (with OLS fit)", fontweight="bold", y=1.01)
save(fig,"e10_socio_scatter.png"); plt.show()


In [ ]:
# Figure 22: variance inflation factors
from sklearn.linear_model import LinearRegression
sd = master[SOCIO].dropna(); vif = {}
for col in SOCIO:
    others = [c for c in SOCIO if c != col]
    r2 = LinearRegression().fit(sd[others], sd[col]).score(sd[others], sd[col])
    vif[SL[col]] = 1/(1-r2) if r2 < 0.999999 else np.inf
vs = pd.Series(vif).sort_values()
fig, ax = plt.subplots(figsize=(8,3.6))
ax.barh(vs.index, vs.values, color=[SECONDARY if v>10 else ("#E69F00" if v>5 else GOOD) for v in vs.values], edgecolor="white")
for i,(k,v) in enumerate(vs.items()): ax.text(v,i,f" {v:.1f}", va="center", fontsize=10)
ax.axvline(5, color="#E69F00", ls="--", lw=1); ax.axvline(10, color=SECONDARY, ls="--", lw=1)
ax.set_title("Variance Inflation Factor (socioeconomic indicators)"); ax.set_xlabel("VIF (dashed at 5 and 10)"); ax.set_ylabel("Indicator")
save(fig,"e11_vif.png"); plt.show()
print("VIF:", {k: round(v,1) for k,v in vif.items()})


In [ ]:
# Figure 24c: socioeconomic profile by cluster.
# Normalised by dividing each indicator by its maximum (share of the largest cluster),
# so the smallest cluster shows its real small value instead of being forced to zero
# (which min-max normalisation would do, making it look like missing data).
prof = master.groupby("Cluster")[SOCIO].mean(); profn = prof / prof.max()
fig, ax = plt.subplots(figsize=(10,5)); x = np.arange(len(prof.index)); w = 0.2
for i, s in enumerate(SOCIO): ax.bar(x+(i-1.5)*w, profn[s], w, label=SL[s], color=OKABE[i])
ax.set_xticks(x); ax.set_xticklabels(prof.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Indicator (share of the largest cluster)"); ax.set_title("Socioeconomic profile by cluster (share of the largest cluster)")
ax.legend(ncol=4, fontsize=8); ax.set_ylim(0,1.08)
save(fig,"e20_socio_profile.png"); plt.show()


## 7. Cluster coverage and completeness (report Figures 7, 24d)

The crime-only panel has eleven clusters; the master panel has ten, because Thohoyandou has no
socioeconomic series and is dropped when the sources are merged on cluster and date. The crime panel
is also a complete balanced grid: every cluster has all 36 crime types in every year, so the
record-count map is uniform and Figure 24d instead shows total Crime Count, which does vary.


In [ ]:
# Figure 7: cluster coverage crime-only vs master
cc_set, cm_set = sorted(crime.Cluster.unique()), sorted(master.Cluster.unique())
only = sorted(set(cc_set) - set(cm_set))
fig, ax = plt.subplots(figsize=(9,3.6))
ax.barh(["Crime-only panel","Master (merged) panel"], [len(cc_set),len(cm_set)], color=[PRIMARY,GOOD], edgecolor="white", height=0.6)
ax.text(len(cc_set),0,f"  {len(cc_set)} clusters", va="center", fontsize=10); ax.text(len(cm_set),1,f"  {len(cm_set)} clusters", va="center", fontsize=10)
ax.set_xlim(0, len(cc_set)+2); ax.set_title("Cluster coverage, crime-only vs master panel"); ax.set_xlabel("Number of municipal clusters")
fig.subplots_adjust(bottom=0.28)
fig.text(0.5, 0.01, f"{', '.join(only)} appears in the crime-only panel but has no socioeconomic series, so it is dropped from the master panel.",
         ha="center", va="bottom", fontsize=9, color=MUTE)
fig.savefig("figures/e14_coverage.png", dpi=150, bbox_inches="tight", facecolor="white"); plt.show()
print("lost in merge:", only)


In [ ]:
# Completeness check: the panel is a COMPLETE balanced grid.
cov = crime.pivot_table(index="Cluster", columns="year", values="Crime Count", aggfunc="count").fillna(0)
print("records per (cluster, year): min", int(cov.values.min()), "max", int(cov.values.max()),
      "-> every cell is identical, so a record-count heatmap would be one flat colour")
print("11 clusters x 36 crime types x 19 years =", 11*36*19, "== panel rows", len(crime), "(no missing combinations)")

# Figure 24d: total Crime Count by cluster and year (this DOES vary, so it is the useful heatmap)
hm = crime.pivot_table(index="Cluster", columns="year", values="Crime Count", aggfunc="sum").fillna(0)
hm = hm.loc[hm.sum(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(11,4.8)); im = ax.imshow(hm.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(hm.columns))); ax.set_xticklabels([int(c) for c in hm.columns], fontsize=8)
ax.set_yticks(range(len(hm.index))); ax.set_yticklabels(hm.index, fontsize=9); ax.grid(False)
ax.set_title("Total Crime Count by cluster and year")
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.set_label("Total Crime Count", fontsize=9)
save(fig,"e22_crime_heatmap.png"); plt.show()


## 8. Feature importance (report Figure 29)

Fitting a Random Forest on the engineered master features shows why enrichment adds little: the
one-year lag of Crime Count dominates, while the four socioeconomic indicators together carry a tiny share.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
mf = pd.read_excel(f"{DATA}/data/05_model_input/master_dataset_features.xlsx")
X = mf.drop(columns=["Crime Count","date"]); y = mf["Crime Count"]
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X, y)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("top feature:", imp.index[0], "=", round(imp.iloc[0]*100,1), "% of importance")
print("socioeconomic block share:", round(imp[[c for c in SOCIO if c in imp.index]].sum()*100, 2), "%")
top18 = imp.head(18)[::-1]
fig, ax = plt.subplots(figsize=(9,6))
ax.barh([c.replace("crime_count_","").replace("_"," ") for c in top18.index], top18.values, color=PRIMARY, edgecolor="white")
ax.set_title("Random Forest feature importance (master condition, top 18)"); ax.set_xlabel("Mean decrease in impurity"); ax.set_ylabel("Feature")
save(fig,"e17_rf_importance.png"); plt.show()


## 9. Summary

The target is a skewed, temporally autocorrelated, spatially concentrated count. Its socioeconomic
candidate predictors are strongly collinear with one another and only weakly and non-linearly related
to it, and in the fitted model the one-year crime lag dominates while the socioeconomic block
contributes little. These findings justify the tree-ensemble model family and the walk-forward
validation, and they anticipate the null result of the enrichment hypothesis test reported in the
main report. Every figure above is saved under `figures/` with the same filename used in the report,
so each report figure has a direct source here.
